In [11]:
pip install yt-dlp requests beautifulsoup4 pydantic

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 12.6 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pydantic]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import json
import time
from typing import Dict, List, Optional
from datetime import datetime, timedelta
import re
from dataclasses import dataclass, asdict
 
import yt_dlp
import requests
from bs4 import BeautifulSoup

In [13]:
def extract_youtube_metadata(youtube_url: str) -> Dict:
    """
    Extract comprehensive metadata from YouTube video using yt-dlp.
    
    Extracted fields:
    - Video ID, Title, Description
    - Duration, Upload date
    - Views, Likes, Comments
    - Creator name, Channel ID
    - Hashtags, Thumbnail URL
    """
    
    ydl_opts = {
        'quiet': True,
        'no_warnings': True,
        'extract_flat': False,
    }
    
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print(f"Extracting metadata from: {youtube_url}")
            info = ydl.extract_info(youtube_url, download=False)
            
            # Parse metadata
            metadata = {
                'platform': 'youtube',
                'video_id': info.get('id'),
                'title': info.get('title'),
                'description': info.get('description'),
                'duration_seconds': info.get('duration'),
                'upload_date': parse_date(info.get('upload_date')),
                'view_count': info.get('view_count'),
                'like_count': info.get('like_count'),
                'comment_count': info.get('comment_count'),
                'creator_name': info.get('uploader'),
                'creator_id': info.get('channel_id'),
                'channel_url': info.get('channel_url'),
                'thumbnail_url': info.get('thumbnail'),
                'video_url': info.get('webpage_url'),
                'hashtags': extract_hashtags(info.get('description', '')),
                'is_live': info.get('is_live', False),
                'is_age_restricted': info.get('age_limit', 0) > 0,
                'extracted_at': datetime.now().isoformat()
            }
            
            return {
                'status': 'success',
                'data': metadata,
                'raw_info_keys': list(info.keys())  # For debugging
            }
            
    except Exception as e:
        return {
            'status': 'error',
            'error_type': type(e).__name__,
            'message': str(e),
            'url': youtube_url
        }
 
def parse_date(date_str: Optional[str]) -> Optional[str]:
    """Convert YYYYMMDD format to ISO format."""
    if not date_str:
        return None
    try:
        date_obj = datetime.strptime(date_str, '%Y%m%d')
        return date_obj.isoformat()
    except:
        return date_str
 
def extract_hashtags(text: str) -> List[str]:
    """Extract hashtags from text."""
    return re.findall(r'#\w+', text)
 
# Test YouTube metadata extraction
print("=" * 80)
print("TEST 1: YouTube Metadata Extraction")
print("=" * 80)
 
youtube_test_urls = [
    'https://www.youtube.com/watch?v=jNQXAC9IVRw',  # First YouTube video
]
 
for url in youtube_test_urls:
    result = extract_youtube_metadata(url)
    
    if result['status'] == 'success':
        data = result['data']
        print(f"\n✅ Metadata extracted successfully")
        print(f"   Title: {data['title']}")
        print(f"   Creator: {data['creator_name']}")
        print(f"   Duration: {data['duration_seconds']} seconds")
        print(f"   Views: {data['view_count']:,}")
        print(f"   Likes: {data['like_count']}")
        print(f"   Comments: {data['comment_count']}")
        print(f"   Upload Date: {data['upload_date']}")
        print(f"   Hashtags: {data['hashtags']}")
        print(f"   Age Restricted: {data['is_age_restricted']}")
    else:
        print(f"\n❌ Error: {result['error_type']}")
        print(f"   Message: {result['message']}")

TEST 1: YouTube Metadata Extraction
Extracting metadata from: https://www.youtube.com/watch?v=jNQXAC9IVRw

✅ Metadata extracted successfully
   Title: Me at the zoo
   Creator: jawed
   Duration: 19 seconds
   Views: 392,778,206
   Likes: 18902578
   Comments: 10000000
   Upload Date: 2005-04-24T00:00:00
   Hashtags: []
   Age Restricted: False


In [14]:
def extract_instagram_metadata_yt_dlp(instagram_url: str) -> Dict:
    """
    Attempt to extract Instagram metadata using yt-dlp.
    Note: Instagram metadata extraction is limited by their anti-scraping measures.
    """
    
    ydl_opts = {
        'quiet': True,
        'no_warnings': True,
    }
    
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print(f"Extracting Instagram metadata from: {instagram_url}")
            info = ydl.extract_info(instagram_url, download=False)
            
            metadata = {
                'platform': 'instagram',
                'video_id': info.get('id'),
                'title': info.get('title'),
                'description': info.get('description'),
                'duration_seconds': info.get('duration'),
                'upload_date': parse_date(info.get('upload_date')),
                'view_count': info.get('view_count'),
                'like_count': info.get('like_count'),
                'comment_count': info.get('comment_count'),
                'thumbnail_url': info.get('thumbnail'),
                'video_url': info.get('webpage_url'),
                'hashtags': extract_hashtags(info.get('description', '')),
                'uploader': info.get('uploader'),
                'uploader_id': info.get('uploader_id'),
                'is_age_restricted': info.get('age_limit', 0) > 0,
                'extracted_at': datetime.now().isoformat()
            }
            
            return {
                'status': 'success',
                'data': metadata
            }
            
    except Exception as e:
        return {
            'status': 'error',
            'error_type': type(e).__name__,
            'message': str(e),
            'url': instagram_url
        }

In [15]:
extract_instagram_metadata_yt_dlp('https://www.instagram.com/reels/DY9AG4GsTu7/')

Extracting Instagram metadata from: https://www.instagram.com/reels/DY9AG4GsTu7/


{'status': 'success',
 'data': {'platform': 'instagram',
  'video_id': 'DY9AG4GsTu7',
  'title': 'Video by dumbteenze',
  'description': '#🇮🇳india is finally being introduced to urbex also known as urban exploring which is popular all around the world for individuals exploring abandoned or haunted or uninhabited places. With content creators such as @dumbteenze revolutionising urbex in India. It is without a doubt urbex becomes popular in the country among individuals hungry for adrenaline.\n.\n.\n.\n.\n.\n#urbex\n#fypppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppp #india #mumbai dumbteenze',
  'duration_seconds': 15.836,
  'upload_date': '2026-05-30T00:00:00',
  'view_count': None,
  'like_count': 39591,
  'comment_count': 130,
  'thumbnail_url': 'https://instagram.fpnq7-2.fna.fbcdn.net/v/t51.71878-15/710748911_962288103293406_4223267256337210672_n.jpg?se=-1&stp=dst-jpegr_e15_tt6&efg=eyJ2ZW5jb2RlX3RhZyI6ImltYWdlX3VybGdlbi42NDAuaGRyLnZpZGVvX2RlZmF1bHRfY292ZXJfZnJh

In [18]:
from pydantic import BaseModel, Field, validator
from typing import Optional
 
class VideoMetadata(BaseModel):
    """Structured metadata for any video."""
    
    platform: str = Field(..., description="youtube or instagram")
    video_id: str = Field(..., description="Unique video identifier")
    title: str
    description: Optional[str] = None
    duration_seconds: int = Field(..., description="Video length in seconds")
    upload_date: Optional[str] = None  # ISO format
    
    # Engagement metrics
    view_count: Optional[int] = None
    like_count: Optional[int] = None
    comment_count: Optional[int] = None
    
    # Creator info
    creator_name: str
    creator_id: Optional[str] = None
    follower_count: Optional[int] = None
    
    # Additional metadata
    hashtags: List[str] = Field(default_factory=list)
    thumbnail_url: Optional[str] = None
    video_url: str
    
    is_live: bool = False
    is_age_restricted: bool = False
    extracted_at: str = Field(default_factory=lambda: datetime.now().isoformat())
    
    @validator('duration_seconds')
    def duration_positive(cls, v):
        if v < 0:
            raise ValueError('Duration must be positive')
        return v
    
    @validator('view_count', 'like_count', 'comment_count', pre=True)
    def positive_counts(cls, v):
        if v is not None and v < 0:
            return None  # Treat negative as missing
        return v
    
    class Config:
        json_schema_extra = {
            'example': {
                'platform': 'youtube',
                'video_id': 'jNQXAC9IVRw',
                'title': 'Me at the zoo',
                'creator_name': 'jawed',
                'view_count': 300000000,
                'like_count': 5000000,
                'comment_count': 1500000,
                'duration_seconds': 18,
                'video_url': 'https://www.youtube.com/watch?v=jNQXAC9IVRw'
            }
        }
 
# Test model validation
print("\n" + "=" * 80)
print("TEST 2: Metadata Models & Validation")
print("=" * 80)
 
test_metadata = {
    'platform': 'youtube',
    'video_id': 'test123',
    'title': 'Test Video',
    'creator_name': 'TestCreator',
    'duration_seconds': 120,
    'view_count': 1000,
    'like_count': 50,
    'comment_count': 10,
    'video_url': 'https://youtube.com/watch?v=test123'
}
 
try:
    validated = VideoMetadata(**test_metadata)
    print("✅ Metadata validation passed")
    print(f"   Model: {validated}")
except Exception as e:
    print(f"❌ Validation error: {e}")


TEST 2: Metadata Models & Validation
✅ Metadata validation passed
   Model: platform='youtube' video_id='test123' title='Test Video' description=None duration_seconds=120 upload_date=None view_count=1000 like_count=50 comment_count=10 creator_name='TestCreator' creator_id=None follower_count=None hashtags=[] thumbnail_url=None video_url='https://youtube.com/watch?v=test123' is_live=False is_age_restricted=False extracted_at='2026-05-31T03:36:08.503742'


/var/folders/8r/_d2fd1c541133bdwx3q28cjr0000gn/T/ipykernel_55051/3383953604.py:33: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator('duration_seconds')
/var/folders/8r/_d2fd1c541133bdwx3q28cjr0000gn/T/ipykernel_55051/3383953604.py:39: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator('view_count', 'like_count', 'comment_count', pre=True)
/var/folders/8r/_d2fd1c541133bdwx3q28cjr0000gn/T/ipykernel_55051/3383953604.p

In [19]:
@dataclass
class EngagementMetrics:
    """Calculate and store engagement metrics."""
    
    video_id: str
    platform: str
    views: int
    likes: int
    comments: int
    
    def __post_init__(self):
        """Validate data after initialization."""
        if self.views <= 0:
            raise ValueError("Views must be positive")
        if self.likes < 0 or self.comments < 0:
            raise ValueError("Likes and comments cannot be negative")
    
    @property
    def engagement_rate(self) -> float:
        """
        Calculate engagement rate.
        Formula: (likes + comments) / views * 100
        """
        if self.views == 0:
            return 0.0
        return ((self.likes + self.comments) / self.views) * 100
    
    @property
    def like_rate(self) -> float:
        """Percentage of viewers who liked."""
        if self.views == 0:
            return 0.0
        return (self.likes / self.views) * 100
    
    @property
    def comment_rate(self) -> float:
        """Percentage of viewers who commented."""
        if self.views == 0:
            return 0.0
        return (self.comments / self.views) * 100
    
    @property
    def comment_to_like_ratio(self) -> float:
        """Ratio of comments to likes."""
        if self.likes == 0:
            return 0.0
        return self.comments / self.likes
    
    @property
    def virality_score(self) -> float:
        """
        Custom virality score (0-100).
        Higher engagement rate + higher like rate = more viral.
        """
        # Normalize metrics to 0-1
        normalized_engagement = min(self.engagement_rate / 5, 1.0)  # 5% is excellent
        normalized_likes = min(self.like_rate / 2, 1.0)  # 2% is excellent
        
        # Weighted average (60% engagement, 40% like rate)
        score = (normalized_engagement * 0.6 + normalized_likes * 0.4) * 100
        return round(score, 2)
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for JSON serialization."""
        return {
            'video_id': self.video_id,
            'platform': self.platform,
            'views': self.views,
            'likes': self.likes,
            'comments': self.comments,
            'engagement_rate': round(self.engagement_rate, 2),
            'like_rate': round(self.like_rate, 2),
            'comment_rate': round(self.comment_rate, 2),
            'comment_to_like_ratio': round(self.comment_to_like_ratio, 2),
            'virality_score': self.virality_score
        }
 
# Test engagement metrics
print("\n" + "=" * 80)
print("TEST 3: Engagement Metrics Calculation")
print("=" * 80)
 
test_videos = [
    {
        'video_id': 'viral_hit',
        'platform': 'youtube',
        'views': 1000000,
        'likes': 50000,
        'comments': 10000
    },
    {
        'video_id': 'moderate',
        'platform': 'youtube',
        'views': 10000,
        'likes': 500,
        'comments': 100
    },
    {
        'video_id': 'low_engagement',
        'platform': 'instagram',
        'views': 5000,
        'likes': 100,
        'comments': 10
    },
]
 
for video in test_videos:
    metrics = EngagementMetrics(**video)
    result = metrics.to_dict()
    
    print(f"\n📊 {result['video_id'].upper()}")
    print(f"   Views: {result['views']:,}")
    print(f"   Likes: {result['likes']:,} ({result['like_rate']:.2f}%)")
    print(f"   Comments: {result['comments']:,} ({result['comment_rate']:.2f}%)")
    print(f"   Engagement Rate: {result['engagement_rate']:.2f}%")
    print(f"   Comment:Like Ratio: {result['comment_to_like_ratio']:.2f}")
    print(f"   Virality Score: {result['virality_score']}/100")
 
# Benchmark: Viral vs Non-viral
print("\n" + "=" * 80)
print("BENCHMARK: Engagement Patterns")
print("=" * 80)
 
engagement_patterns = {
    'mega_viral': {'views': 100000000, 'likes': 5000000, 'comments': 2000000},
    'viral': {'views': 10000000, 'likes': 1000000, 'comments': 500000},
    'popular': {'views': 1000000, 'likes': 100000, 'comments': 50000},
    'average': {'views': 100000, 'likes': 5000, 'comments': 500},
    'low': {'views': 10000, 'likes': 500, 'comments': 50},
}
 
results = []
for category, data in engagement_patterns.items():
    metrics = EngagementMetrics(
        video_id=category,
        platform='youtube',
        views=data['views'],
        likes=data['likes'],
        comments=data['comments']
    )
    results.append({
        'category': category,
        'engagement_rate': metrics.engagement_rate,
        'virality_score': metrics.virality_score
    })
 
for r in results:
    bar = '█' * int(r['virality_score'] / 5)
    print(f"{r['category']:15} | {r['engagement_rate']:6.2f}% | Virality: {bar}")


TEST 3: Engagement Metrics Calculation

📊 VIRAL_HIT
   Views: 1,000,000
   Likes: 50,000 (5.00%)
   Comments: 10,000 (1.00%)
   Engagement Rate: 6.00%
   Comment:Like Ratio: 0.20
   Virality Score: 100.0/100

📊 MODERATE
   Views: 10,000
   Likes: 500 (5.00%)
   Comments: 100 (1.00%)
   Engagement Rate: 6.00%
   Comment:Like Ratio: 0.20
   Virality Score: 100.0/100

📊 LOW_ENGAGEMENT
   Views: 5,000
   Likes: 100 (2.00%)
   Comments: 10 (0.20%)
   Engagement Rate: 2.20%
   Comment:Like Ratio: 0.10
   Virality Score: 66.4/100

BENCHMARK: Engagement Patterns
mega_viral      |   7.00% | Virality: ████████████████████
viral           |  15.00% | Virality: ████████████████████
popular         |  15.00% | Virality: ████████████████████
average         |   5.50% | Virality: ████████████████████
low             |   5.50% | Virality: ████████████████████


In [22]:
# Sample URLs (real YouTube videos + sample Instagram reels)
youtube_urls = [
    "https://www.youtube.com/watch?v=jNQXAC9IVRw",  # Me at the zoo
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",  # Never Gonna Give You Up
    "https://www.youtube.com/watch?v=9bZkp7q19f0",  # Gangnam Style
]
instagram_urls = [
    "https://www.instagram.com/reel/DY9AG4GsTu7/",
    "https://www.instagram.com/reels/DY6pGBLMfbe/"
]

def _print_youtube_summary(url: str) -> None:
    result = extract_youtube_metadata(url)
    if result.get("status") != "success":
        print(f"YouTube metadata error for {url}: {result.get('message')}")
        return
    data = result["data"]
    print("\n---")
    print(f"Title: {data.get('title')}")
    print(f"Creator: {data.get('creator_name')}")
    print(f"Views: {data.get('view_count')}")
    print(f"Likes: {data.get('like_count')}")
    print(f"Comments: {data.get('comment_count')}")
    print(f"Upload Date: {data.get('upload_date')}")
    print(f"Hashtags: {data.get('hashtags')}")

    views = data.get("view_count") or 0
    likes = data.get("like_count") or 0
    comments = data.get("comment_count") or 0
    if views > 0:
        metrics = EngagementMetrics(
            video_id=data.get("video_id") or "unknown",
            platform="youtube",
            views=views,
            likes=likes,
            comments=comments,
        )
        metrics_dict = metrics.to_dict()
        print(f"Engagement Rate: {metrics_dict['engagement_rate']}%")
        print(f"Like Rate: {metrics_dict['like_rate']}%")
        print(f"Comment Rate: {metrics_dict['comment_rate']}%")
        print(f"Virality Score: {metrics_dict['virality_score']}")
    else:
        print("Engagement analytics skipped (missing view count).")

def _print_instagram_summary(url: str) -> None:
    result = extract_instagram_metadata_yt_dlp(url)
    if result.get("status") != "success":
        print(f"Instagram metadata error for {url}: {result.get('message')}")
        return
    data = result.get("data", {})
    print("\n---")
    print(f"Video URL: {data.get('video_url')}")
    print(f"Title: {data.get('title')}")
    print(f"Views: {data.get('view_count')}")
    print(f"Likes: {data.get('like_count')}")
    print(f"Comments: {data.get('comment_count')}")

print("=" * 80)
print("TEST 4: Sample Videos + Metadata Analytics")
print("=" * 80)
for url in youtube_urls:
    _print_youtube_summary(url)

print("\n" + "=" * 80)
print("TEST 5: Instagram Metadata (Best Effort)")
print("=" * 80)
for url in instagram_urls:
    _print_instagram_summary(url)

TEST 4: Sample Videos + Metadata Analytics
Extracting metadata from: https://www.youtube.com/watch?v=jNQXAC9IVRw

---
Title: Me at the zoo
Creator: jawed
Views: 392779712
Likes: 18902636
Comments: 10000000
Upload Date: 2005-04-24T00:00:00
Hashtags: []
Engagement Rate: 7.36%
Like Rate: 4.81%
Comment Rate: 2.55%
Virality Score: 100.0
Extracting metadata from: https://www.youtube.com/watch?v=dQw4w9WgXcQ

---
Title: Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)
Creator: Rick Astley
Views: 1777876148
Likes: 19129065
Comments: 2400000
Upload Date: 2009-10-25T00:00:00
Hashtags: ['#RickAstleyNever', '#RickAstley', '#NeverGonnaGiveYouUp', '#WheneverYouNeedSomebody', '#OfficialMusicVideo']
Engagement Rate: 1.21%
Like Rate: 1.08%
Comment Rate: 0.13%
Virality Score: 36.05
Extracting metadata from: https://www.youtube.com/watch?v=9bZkp7q19f0

---
Title: PSY - GANGNAM STYLE(강남스타일) M/V
Creator: officialpsy
Views: 5953793257
Likes: 31858992
Comments: 5400000
Upload Date: 2012-07